# Notebook 2 — Ensemble RoBERTa-large-BNE + TF-IDF multivista MLP

Este notebook entrena dos redes neuronales y combina probabilidades:

1. RoBERTa-large-BNE + MLP potente.
2. TF-IDF multivista `char + char_wb + word` + MLP sparse.

La métrica principal para escoger checkpoints y peso del ensemble es **accuracy**.

In [ ]:
# =========================
# Instalación del ambiente
# =========================
# Ejecuta esta celda al inicio del notebook. Después de instalar, reinicia el kernel.

%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
%pip install transformers accelerate sentencepiece pandas numpy scikit-learn matplotlib tqdm scipy


In [ ]:
# =========================
# Imports y configuración base
# =========================

import os
import gc
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("CONFIGURACIÓN DEL ENTORNO")
print("=" * 70)
print("Python OK")
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA PyTorch:", torch.version.cuda)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("=" * 70)


In [ ]:
# =========================
# Limpieza segura de memoria
# =========================

def clean_memory(extra_vars=None):
    extra_vars = extra_vars or []
    for name in extra_vars:
        if name in globals():
            try:
                del globals()[name]
            except Exception:
                pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("GPU asignada:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
        print("GPU reservada:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

clean_memory()


## 1. Configuración

In [ ]:
# =========================
# Configuración general del ensemble
# =========================

TRAIN_PATH = "train.csv"
EVAL_PATH = "eval.csv"

TEXT_COL = "text"
LABEL_COL = "decade"

MODEL_NAME = "PlanTL-GOB-ES/roberta-large-bne"

MAX_LEN = 512
MAX_TEXT_CHARS = 3000

EPOCHS_ROBERTA = 30
EPOCHS_TFIDF = 30
PATIENCE = 2

BATCH_SIZE_ROBERTA = 16 if DEVICE.type == "cuda" else 4
BATCH_SIZE_TFIDF = 256 if DEVICE.type == "cuda" else 128
GRAD_ACCUM_STEPS = 1

LR_ROBERTA = 1e-5
LR_ROBERTA_MLP = 2e-4
LR_TFIDF = 2e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.08
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0

ROBERTA_MLP_H1 = 2048
ROBERTA_MLP_H2 = 1024
ROBERTA_MLP_H3 = 512

TFIDF_H1 = 1024
TFIDF_H2 = 512
TFIDF_H3 = 256

DROPOUT = 0.35

CHAR_NGRAM_RANGE = (2, 6)
CHAR_WB_NGRAM_RANGE = (3, 6)
WORD_NGRAM_RANGE = (1, 2)

CHAR_MAX_FEATURES = 180_000
CHAR_WB_MAX_FEATURES = 120_000
WORD_MAX_FEATURES = 80_000
MIN_DF = 2
MAX_DF = 1.0

CHECKPOINT_DIR = Path("checkpoints_roberta_large_tfidf_mlp_ensemble")
CHECKPOINT_DIR.mkdir(exist_ok=True)

BEST_ROBERTA_PATH = CHECKPOINT_DIR / "best_roberta_large_mlp.pt"
BEST_TFIDF_PATH = CHECKPOINT_DIR / "best_tfidf_multiview_mlp.pt"

print("Modelo:", MODEL_NAME)
print("Patience:", PATIENCE)


In [ ]:
# =========================
# Configuración
# =========================

TRAIN_PATH = "train.csv"
EVAL_PATH = "eval.csv"

TEXT_COL = "text"
LABEL_COL = "decade"

MODEL_NAME = "PlanTL-GOB-ES/roberta-large-bne"

MAX_LEN = 512
MAX_TEXT_CHARS = 3000

EPOCHS = 30
PATIENCE = 2

BATCH_SIZE = 16 if DEVICE.type == "cuda" else 4
GRAD_ACCUM_STEPS = 1

LR_ROBERTA = 1e-5
LR_MLP = 2e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.08
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0

MLP_HIDDEN_1 = 2048
MLP_HIDDEN_2 = 1024
MLP_HIDDEN_3 = 512
DROPOUT = 0.35

CHECKPOINT_DIR = Path("checkpoints_roberta_large_mlp")
CHECKPOINT_DIR.mkdir(exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_roberta_large_mlp.pt"
LAST_MODEL_PATH = CHECKPOINT_DIR / "last_roberta_large_mlp.pt"

print("Modelo:", MODEL_NAME)
print("Batch size:", BATCH_SIZE)
print("Max len:", MAX_LEN)
print("Patience:", PATIENCE)


## 2. Carga de datos y mapeo de etiquetas

El mapeo se revisa explícitamente para evitar errores con las clases extremas `150` y `188`.

In [ ]:
# =========================
# Cargar train.csv
# =========================

df = pd.read_csv(TRAIN_PATH)

assert TEXT_COL in df.columns, f"Falta columna {TEXT_COL}"
assert LABEL_COL in df.columns, f"Falta columna {LABEL_COL}"

df = df[[TEXT_COL, LABEL_COL]].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

decades = sorted(df[LABEL_COL].unique())
decade_to_idx = {decade: idx for idx, decade in enumerate(decades)}
idx_to_decade = {idx: decade for decade, idx in decade_to_idx.items()}

df["label_idx"] = df[LABEL_COL].map(decade_to_idx)

NUM_CLASSES = len(decades)

assert decades[0] == 150, f"La primera década debería ser 150, pero es {decades[0]}"
assert decades[-1] == 188, f"La última década debería ser 188, pero es {decades[-1]}"
assert NUM_CLASSES == 39, f"Esperábamos 39 clases, pero hay {NUM_CLASSES}"
assert decade_to_idx[150] == 0
assert decade_to_idx[188] == 38
assert idx_to_decade[0] == 150
assert idx_to_decade[38] == 188

print("Shape:", df.shape)
print("Clases:", NUM_CLASSES)
print("Mapping extremo:", {150: decade_to_idx[150], 188: decade_to_idx[188]})
display(df.head())
display(df[LABEL_COL].value_counts().sort_index())


In [ ]:
# =========================
# Split train / val / test
# =========================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_idx"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = part[LABEL_COL].value_counts().sort_index()
    print(name, "150:", counts.get(150, 0), "188:", counts.get(188, 0))


## 2. Tokenizer y datasets para RoBERTa

In [ ]:
# =========================
# Tokenizer RoBERTa
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def crop_train_text(text, max_chars=MAX_TEXT_CHARS):
    text = str(text)
    if len(text) <= max_chars:
        return text
    start = random.randint(0, len(text) - max_chars)
    return text[start:start + max_chars]

def crop_eval_text(text, max_chars=MAX_TEXT_CHARS, mode="center"):
    text = str(text)
    if len(text) <= max_chars:
        return text
    if mode == "start":
        return text[:max_chars]
    if mode == "end":
        return text[-max_chars:]
    start = (len(text) - max_chars) // 2
    return text[start:start + max_chars]

def get_eval_crops(text, max_chars=MAX_TEXT_CHARS):
    text = str(text)
    if len(text) <= max_chars:
        return [text]
    crops = [
        text[:max_chars],
        text[(len(text)-max_chars)//2:(len(text)-max_chars)//2+max_chars],
        text[-max_chars:],
    ]
    out, seen = [], set()
    for c in crops:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

class TextDataset(Dataset):
    def __init__(self, dataframe, train_mode=False):
        self.texts = dataframe[TEXT_COL].astype(str).tolist()
        self.labels = dataframe["label_idx"].astype(int).tolist()
        self.train_mode = train_mode

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        text = crop_train_text(text) if self.train_mode else crop_eval_text(text)
        return {"text": text, "label": self.labels[idx]}

def collate_text_batch(batch):
    texts = [x["text"] for x in batch]
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt")
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels, "texts": texts}

train_text_dataset = TextDataset(train_df, train_mode=True)
val_text_dataset = TextDataset(val_df, train_mode=False)
test_text_dataset = TextDataset(test_df, train_mode=False)

train_text_loader = DataLoader(train_text_dataset, batch_size=BATCH_SIZE_ROBERTA, shuffle=True, collate_fn=collate_text_batch, num_workers=0, pin_memory=(DEVICE.type=="cuda"))
val_text_loader = DataLoader(val_text_dataset, batch_size=BATCH_SIZE_ROBERTA, shuffle=False, collate_fn=collate_text_batch, num_workers=0, pin_memory=(DEVICE.type=="cuda"))
test_text_loader = DataLoader(test_text_dataset, batch_size=BATCH_SIZE_ROBERTA, shuffle=False, collate_fn=collate_text_batch, num_workers=0, pin_memory=(DEVICE.type=="cuda"))

print(len(train_text_loader), len(val_text_loader), len(test_text_loader))


## 3. Modelo 1: RoBERTa-large-BNE + MLP

In [ ]:
# =========================
# RoBERTa + MLP
# =========================

class RobertaLargeMLPClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden = self.roberta.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden, ROBERTA_MLP_H1),
            nn.LayerNorm(ROBERTA_MLP_H1),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(ROBERTA_MLP_H1, ROBERTA_MLP_H2),
            nn.LayerNorm(ROBERTA_MLP_H2),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(ROBERTA_MLP_H2, ROBERTA_MLP_H3),
            nn.LayerNorm(ROBERTA_MLP_H3),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(ROBERTA_MLP_H3, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)

roberta_model = RobertaLargeMLPClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)

try:
    roberta_model.roberta.gradient_checkpointing_enable()
except Exception:
    pass

print("RoBERTa params:", sum(p.numel() for p in roberta_model.parameters()))


In [ ]:
# =========================
# Métricas comunes
# =========================

idx_to_decade_array = np.array([idx_to_decade[i] for i in range(NUM_CLASSES)])

def compute_metrics_np(y_true_idx, y_pred_idx):
    y_true_idx = np.asarray(y_true_idx)
    y_pred_idx = np.asarray(y_pred_idx)
    true_dec = idx_to_decade_array[y_true_idx]
    pred_dec = idx_to_decade_array[y_pred_idx]
    abs_err = np.abs(true_dec - pred_dec)
    return {
        "acc": float(np.mean(y_true_idx == y_pred_idx)),
        "mae": float(np.mean(abs_err)),
        "acc_pm1": float(np.mean(abs_err <= 1)),
        "acc_pm2": float(np.mean(abs_err <= 2)),
    }


In [ ]:
# =========================
# Entrenar RoBERTa
# =========================

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

roberta_params, head_params = [], []
for name, param in roberta_model.named_parameters():
    if name.startswith("roberta."):
        roberta_params.append(param)
    else:
        head_params.append(param)

roberta_optimizer = torch.optim.AdamW(
    [
        {"params": roberta_params, "lr": LR_ROBERTA, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": LR_ROBERTA_MLP, "weight_decay": WEIGHT_DECAY},
    ]
)

updates_per_epoch = math.ceil(len(train_text_loader) / GRAD_ACCUM_STEPS)
total_steps = updates_per_epoch * EPOCHS_ROBERTA
warmup_steps = int(total_steps * WARMUP_RATIO)

roberta_scheduler = get_linear_schedule_with_warmup(roberta_optimizer, warmup_steps, total_steps)
roberta_scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type=="cuda"))

def train_roberta_epoch():
    roberta_model.train()
    roberta_optimizer.zero_grad(set_to_none=True)
    total_loss, total_n = 0.0, 0
    all_preds, all_labels = [], []

    for step, batch in enumerate(tqdm(train_text_loader, desc="Train RoBERTa", leave=False), start=1):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = roberta_model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss_back = loss / GRAD_ACCUM_STEPS

        roberta_scaler.scale(loss_back).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_text_loader):
            roberta_scaler.unscale_(roberta_optimizer)
            torch.nn.utils.clip_grad_norm_(roberta_model.parameters(), GRAD_CLIP)
            roberta_scaler.step(roberta_optimizer)
            roberta_scaler.update()
            roberta_optimizer.zero_grad(set_to_none=True)
            roberta_scheduler.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        all_preds.extend(torch.argmax(logits.detach(), dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    m = compute_metrics_np(all_labels, all_preds)
    m["loss"] = total_loss / total_n
    return m

@torch.no_grad()
def evaluate_roberta_loader(loader, desc="Eval RoBERTa"):
    roberta_model.eval()
    total_loss, total_n = 0.0, 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc=desc, leave=False):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = roberta_model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    m = compute_metrics_np(all_labels, all_preds)
    m["loss"] = total_loss / total_n
    return m

roberta_history = []
best_roberta_acc = -1
bad = 0

for epoch in range(1, EPOCHS_ROBERTA + 1):
    print(f"\nRoBERTa epoch {epoch}/{EPOCHS_ROBERTA}")
    tr = train_roberta_epoch()
    va = evaluate_roberta_loader(val_text_loader, "Val RoBERTa")

    row = {"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}}
    roberta_history.append(row)

    print("train acc:", tr["acc"], "val acc:", va["acc"], "val mae:", va["mae"])

    if va["acc"] > best_roberta_acc:
        best_roberta_acc = va["acc"]
        bad = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": roberta_model.state_dict(),
            "best_val_acc": best_roberta_acc,
            "history": roberta_history,
            "decade_to_idx": decade_to_idx,
            "idx_to_decade": idx_to_decade,
            "decades": decades,
        }, BEST_ROBERTA_PATH)
        print("Guardado mejor RoBERTa.")
    else:
        bad += 1
        print("Sin mejora:", bad)

    if bad >= PATIENCE:
        print("Early stopping RoBERTa.")
        break

    clean_memory()

print("Best RoBERTa val acc:", best_roberta_acc)


## 4. Modelo 2: TF-IDF multivista + MLP sparse

In [ ]:
# =========================
# TF-IDF multivista
# =========================

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=CHAR_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32,
)

char_wb_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=CHAR_WB_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_WB_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32,
)

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=WORD_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=WORD_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    token_pattern=r"(?u)\b\w+\b",
    dtype=np.float32,
)

train_texts = train_df[TEXT_COL].tolist()
val_texts = val_df[TEXT_COL].tolist()
test_texts = test_df[TEXT_COL].tolist()

X_train = sp.hstack([
    char_vectorizer.fit_transform(train_texts),
    char_wb_vectorizer.fit_transform(train_texts),
    word_vectorizer.fit_transform(train_texts),
], format="csr", dtype=np.float32)

X_val = sp.hstack([
    char_vectorizer.transform(val_texts),
    char_wb_vectorizer.transform(val_texts),
    word_vectorizer.transform(val_texts),
], format="csr", dtype=np.float32)

X_test = sp.hstack([
    char_vectorizer.transform(test_texts),
    char_wb_vectorizer.transform(test_texts),
    word_vectorizer.transform(test_texts),
], format="csr", dtype=np.float32)

y_train = train_df["label_idx"].values.astype(np.int64)
y_val = val_df["label_idx"].values.astype(np.int64)
y_test = test_df["label_idx"].values.astype(np.int64)

INPUT_DIM = X_train.shape[1]
print(X_train.shape, X_val.shape, X_test.shape)


In [ ]:
# =========================
# Sparse dataset y utilidades
# =========================

class SparseIndexDataset(Dataset):
    def __init__(self, y):
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {"idx": idx, "label": self.y[idx]}

def sparse_collate_fn(batch):
    return {
        "indices": torch.tensor([x["idx"] for x in batch], dtype=torch.long),
        "labels": torch.stack([x["label"] for x in batch]),
    }

def csr_batch_to_torch_sparse(X_csr_batch, device=DEVICE):
    coo = X_csr_batch.tocoo()
    indices = torch.tensor(np.vstack((coo.row, coo.col)), dtype=torch.long)
    values = torch.tensor(coo.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, values, torch.Size(coo.shape), dtype=torch.float32).coalesce().to(device)

tfidf_train_loader = DataLoader(SparseIndexDataset(y_train), batch_size=BATCH_SIZE_TFIDF, shuffle=True, collate_fn=sparse_collate_fn, num_workers=0)
tfidf_val_loader = DataLoader(SparseIndexDataset(y_val), batch_size=BATCH_SIZE_TFIDF, shuffle=False, collate_fn=sparse_collate_fn, num_workers=0)
tfidf_test_loader = DataLoader(SparseIndexDataset(y_test), batch_size=BATCH_SIZE_TFIDF, shuffle=False, collate_fn=sparse_collate_fn, num_workers=0)


In [ ]:
# =========================
# TF-IDF MLP sparse
# =========================

class SparseTfidfMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc1_weight = nn.Parameter(torch.empty(TFIDF_H1, input_dim))
        self.fc1_bias = nn.Parameter(torch.zeros(TFIDF_H1))

        self.norm1 = nn.LayerNorm(TFIDF_H1)
        self.drop1 = nn.Dropout(DROPOUT)

        self.fc2 = nn.Linear(TFIDF_H1, TFIDF_H2)
        self.norm2 = nn.LayerNorm(TFIDF_H2)
        self.drop2 = nn.Dropout(DROPOUT)

        self.fc3 = nn.Linear(TFIDF_H2, TFIDF_H3)
        self.norm3 = nn.LayerNorm(TFIDF_H3)
        self.drop3 = nn.Dropout(DROPOUT)

        self.out = nn.Linear(TFIDF_H3, num_classes)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fc1_weight)
        nn.init.zeros_(self.fc1_bias)
        for m in [self.fc2, self.fc3, self.out]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x_sparse):
        x = torch.sparse.mm(x_sparse, self.fc1_weight.t()) + self.fc1_bias
        x = self.drop1(F.gelu(self.norm1(x)))
        x = self.drop2(F.gelu(self.norm2(self.fc2(x))))
        x = self.drop3(F.gelu(self.norm3(self.fc3(x))))
        return self.out(x)

tfidf_model = SparseTfidfMLP(INPUT_DIM, NUM_CLASSES).to(DEVICE)
print("TFIDF params:", sum(p.numel() for p in tfidf_model.parameters()))


In [ ]:
# =========================
# Entrenar TF-IDF MLP
# =========================

tfidf_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
tfidf_optimizer = torch.optim.AdamW(tfidf_model.parameters(), lr=LR_TFIDF, weight_decay=WEIGHT_DECAY)
tfidf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(tfidf_optimizer, mode="max", factor=0.5, patience=1)

def train_tfidf_epoch():
    tfidf_model.train()
    total_loss, total_n = 0.0, 0
    all_preds, all_labels = [], []

    for batch in tqdm(tfidf_train_loader, desc="Train TFIDF", leave=False):
        idx = batch["indices"].numpy()
        labels = batch["labels"].to(DEVICE)
        Xb = csr_batch_to_torch_sparse(X_train[idx], DEVICE)

        tfidf_optimizer.zero_grad(set_to_none=True)
        logits = tfidf_model(Xb)
        loss = tfidf_criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tfidf_model.parameters(), GRAD_CLIP)
        tfidf_optimizer.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        all_preds.extend(torch.argmax(logits.detach(), dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    m = compute_metrics_np(all_labels, all_preds)
    m["loss"] = total_loss / total_n
    return m

@torch.no_grad()
def evaluate_tfidf_loader(loader, Xmat, desc="Eval TFIDF"):
    tfidf_model.eval()
    total_loss, total_n = 0.0, 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc=desc, leave=False):
        idx = batch["indices"].numpy()
        labels = batch["labels"].to(DEVICE)
        Xb = csr_batch_to_torch_sparse(Xmat[idx], DEVICE)
        logits = tfidf_model(Xb)
        loss = tfidf_criterion(logits, labels)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    m = compute_metrics_np(all_labels, all_preds)
    m["loss"] = total_loss / total_n
    return m

tfidf_history = []
best_tfidf_acc = -1
bad = 0

for epoch in range(1, EPOCHS_TFIDF + 1):
    print(f"\nTFIDF epoch {epoch}/{EPOCHS_TFIDF}")
    tr = train_tfidf_epoch()
    va = evaluate_tfidf_loader(tfidf_val_loader, X_val, "Val TFIDF")
    tfidf_scheduler.step(va["acc"])

    row = {"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}}
    tfidf_history.append(row)

    print("train acc:", tr["acc"], "val acc:", va["acc"], "val mae:", va["mae"])

    if va["acc"] > best_tfidf_acc:
        best_tfidf_acc = va["acc"]
        bad = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": tfidf_model.state_dict(),
            "best_val_acc": best_tfidf_acc,
            "history": tfidf_history,
            "vectorizers": {"char": char_vectorizer, "char_wb": char_wb_vectorizer, "word": word_vectorizer},
            "decade_to_idx": decade_to_idx,
            "idx_to_decade": idx_to_decade,
            "decades": decades,
            "input_dim": INPUT_DIM,
        }, BEST_TFIDF_PATH)
        print("Guardado mejor TFIDF.")
    else:
        bad += 1
        print("Sin mejora:", bad)

    if bad >= PATIENCE:
        print("Early stopping TFIDF.")
        break

    clean_memory()

print("Best TFIDF val acc:", best_tfidf_acc)


## 5. Ensemble en validación, test y submission normal

In [ ]:
# =========================
# Cargar mejores modelos
# =========================

clean_memory(["roberta_optimizer", "roberta_scheduler", "roberta_scaler", "tfidf_optimizer", "tfidf_scheduler"])

rob_ckpt = torch.load(BEST_ROBERTA_PATH, map_location="cpu", weights_only=False)
roberta_model.load_state_dict(rob_ckpt["model_state_dict"])
roberta_model.to(DEVICE)
roberta_model.eval()

tfidf_ckpt = torch.load(BEST_TFIDF_PATH, map_location="cpu", weights_only=False)
tfidf_model.load_state_dict(tfidf_ckpt["model_state_dict"])
tfidf_model.to(DEVICE)
tfidf_model.eval()

print("RoBERTa best epoch:", rob_ckpt["epoch"], "acc:", rob_ckpt["best_val_acc"])
print("TFIDF best epoch:", tfidf_ckpt["epoch"], "acc:", tfidf_ckpt["best_val_acc"])


In [ ]:
# =========================
# Probabilidades RoBERTa y TFIDF
# =========================

@torch.no_grad()
def predict_roberta_probs_texts(texts, batch_size=BATCH_SIZE_ROBERTA):
    roberta_model.eval()
    probs_out = []

    for text in tqdm(texts, desc="RoBERTa probs"):
        crops = get_eval_crops(text)
        crop_probs = []

        for start in range(0, len(crops), batch_size):
            batch_texts = crops[start:start+batch_size]
            enc = tokenizer(batch_texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt")
            input_ids = enc["input_ids"].to(DEVICE)
            attention_mask = enc["attention_mask"].to(DEVICE)

            with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
                logits = roberta_model(input_ids, attention_mask)
                probs = torch.softmax(logits, dim=1)

            crop_probs.append(probs.detach().cpu())

        probs_out.append(torch.cat(crop_probs, dim=0).mean(dim=0).numpy())

    return np.vstack(probs_out)

@torch.no_grad()
def predict_tfidf_probs(Xmat, batch_size=BATCH_SIZE_TFIDF):
    tfidf_model.eval()
    probs_out = []
    indices = np.arange(Xmat.shape[0])

    for start in tqdm(range(0, len(indices), batch_size), desc="TFIDF probs"):
        idx = indices[start:start+batch_size]
        Xb = csr_batch_to_torch_sparse(Xmat[idx], DEVICE)
        logits = tfidf_model(Xb)
        probs = torch.softmax(logits, dim=1)
        probs_out.append(probs.detach().cpu().numpy())

    return np.vstack(probs_out)

val_roberta_probs = predict_roberta_probs_texts(val_df[TEXT_COL].tolist())
test_roberta_probs = predict_roberta_probs_texts(test_df[TEXT_COL].tolist())

val_tfidf_probs = predict_tfidf_probs(X_val)
test_tfidf_probs = predict_tfidf_probs(X_test)


In [ ]:
# =========================
# Buscar mejor peso del ensemble por validation accuracy
# =========================

y_val = val_df["label_idx"].values
y_test = test_df["label_idx"].values

best_w = None
best_acc = -1
records = []

for w in np.linspace(0, 1, 21):
    # w = peso RoBERTa
    val_probs = w * val_roberta_probs + (1 - w) * val_tfidf_probs
    pred = val_probs.argmax(axis=1)
    m = compute_metrics_np(y_val, pred)
    records.append({"w_roberta": w, **m})

    if m["acc"] > best_acc:
        best_acc = m["acc"]
        best_w = w

weights_df = pd.DataFrame(records)
display(weights_df)

print("Mejor peso RoBERTa:", best_w)
print("Mejor val acc ensemble:", best_acc)

test_probs = best_w * test_roberta_probs + (1 - best_w) * test_tfidf_probs
test_pred = test_probs.argmax(axis=1)
test_metrics = compute_metrics_np(y_test, test_pred)

print("TEST ensemble:", test_metrics)


In [ ]:
# =========================
# Matriz de confusión ensemble
# =========================

cm = confusion_matrix(y_test, test_pred, labels=list(range(NUM_CLASSES)))

cm_df = pd.DataFrame(
    cm,
    index=[idx_to_decade[i] for i in range(NUM_CLASSES)],
    columns=[idx_to_decade[i] for i in range(NUM_CLASSES)]
)

display(cm_df)

plt.figure(figsize=(15, 12))
plt.imshow(cm, aspect="auto")
plt.title("Matriz de confusión sin normalizar - Ensemble")
plt.xlabel("Década predicha")
plt.ylabel("Década real")
plt.xticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)], rotation=90)
plt.yticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)])
plt.colorbar(label="Conteo")
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# Submission normal ensemble
# =========================

SUBMISSION_PATH = "submission_roberta_large_tfidf_ensemble_validated.csv"

eval_df = pd.read_csv(EVAL_PATH)
assert "id" in eval_df.columns and "text" in eval_df.columns
eval_df["text"] = eval_df["text"].fillna("").astype(str)

eval_texts = eval_df[TEXT_COL].tolist()

# Transformar eval para TFIDF
X_eval = sp.hstack([
    char_vectorizer.transform(eval_texts),
    char_wb_vectorizer.transform(eval_texts),
    word_vectorizer.transform(eval_texts),
], format="csr", dtype=np.float32)

eval_roberta_probs = predict_roberta_probs_texts(eval_texts)
eval_tfidf_probs = predict_tfidf_probs(X_eval)

eval_probs = best_w * eval_roberta_probs + (1 - best_w) * eval_tfidf_probs
eval_pred = eval_probs.argmax(axis=1)

answers = [idx_to_decade[int(i)] for i in eval_pred]

submission = pd.DataFrame({"id": eval_df["id"].values, "answer": answers})
submission.to_csv(SUBMISSION_PATH, index=False)

print("Guardado:", SUBMISSION_PATH)
display(submission.head())
display(submission["answer"].value_counts().sort_index())


## 6. Entrenamiento final full-data y submission final ensemble

In [ ]:
# =========================
# Configuración full-data
# =========================

FINAL_EPOCHS_ROBERTA = int(rob_ckpt["epoch"])
FINAL_EPOCHS_TFIDF = int(tfidf_ckpt["epoch"])

FINAL_DIR = Path("checkpoints_roberta_large_tfidf_ensemble_full_data")
FINAL_DIR.mkdir(exist_ok=True)

FINAL_ROBERTA_PATH = FINAL_DIR / "final_roberta_large.pt"
FINAL_TFIDF_PATH = FINAL_DIR / "final_tfidf_mlp.pt"

print("Final epochs RoBERTa:", FINAL_EPOCHS_ROBERTA)
print("Final epochs TFIDF:", FINAL_EPOCHS_TFIDF)
print("Ensemble weight RoBERTa:", best_w)


In [ ]:
# =========================
# Full-data RoBERTa
# =========================

clean_memory(["roberta_model", "tfidf_model"])

full_text_dataset = TextDataset(df.reset_index(drop=True), train_mode=True)
full_text_loader = DataLoader(full_text_dataset, batch_size=BATCH_SIZE_ROBERTA, shuffle=True, collate_fn=collate_text_batch, num_workers=0, pin_memory=(DEVICE.type=="cuda"))

final_roberta = RobertaLargeMLPClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)
try:
    final_roberta.roberta.gradient_checkpointing_enable()
except Exception:
    pass

fr_params, fh_params = [], []
for name, param in final_roberta.named_parameters():
    if name.startswith("roberta."):
        fr_params.append(param)
    else:
        fh_params.append(param)

final_roberta_optimizer = torch.optim.AdamW(
    [
        {"params": fr_params, "lr": LR_ROBERTA, "weight_decay": WEIGHT_DECAY},
        {"params": fh_params, "lr": LR_ROBERTA_MLP, "weight_decay": WEIGHT_DECAY},
    ]
)

steps = math.ceil(len(full_text_loader) / GRAD_ACCUM_STEPS) * FINAL_EPOCHS_ROBERTA
warmup = int(steps * WARMUP_RATIO)
final_roberta_scheduler = get_linear_schedule_with_warmup(final_roberta_optimizer, warmup, steps)
final_roberta_scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type=="cuda"))

def train_final_roberta_epoch():
    final_roberta.train()
    final_roberta_optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(tqdm(full_text_loader, desc="Final RoBERTa", leave=False), start=1):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
            logits = final_roberta(input_ids, attention_mask)
            loss = criterion(logits, labels) / GRAD_ACCUM_STEPS

        final_roberta_scaler.scale(loss).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(full_text_loader):
            final_roberta_scaler.unscale_(final_roberta_optimizer)
            torch.nn.utils.clip_grad_norm_(final_roberta.parameters(), GRAD_CLIP)
            final_roberta_scaler.step(final_roberta_optimizer)
            final_roberta_scaler.update()
            final_roberta_optimizer.zero_grad(set_to_none=True)
            final_roberta_scheduler.step()

for epoch in range(1, FINAL_EPOCHS_ROBERTA + 1):
    print(f"Final RoBERTa epoch {epoch}/{FINAL_EPOCHS_ROBERTA}")
    train_final_roberta_epoch()
    clean_memory()

torch.save({
    "epoch": FINAL_EPOCHS_ROBERTA,
    "model_state_dict": final_roberta.state_dict(),
    "decade_to_idx": decade_to_idx,
    "idx_to_decade": idx_to_decade,
    "decades": decades,
}, FINAL_ROBERTA_PATH)

print("Guardado:", FINAL_ROBERTA_PATH)


In [ ]:
# =========================
# Full-data TFIDF
# =========================

full_texts = df[TEXT_COL].fillna("").astype(str).tolist()
full_y = df["label_idx"].values.astype(np.int64)

final_char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=CHAR_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32,
)
final_char_wb_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=CHAR_WB_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_WB_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32,
)
final_word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=WORD_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=WORD_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    token_pattern=r"(?u)\b\w+\b",
    dtype=np.float32,
)

X_full = sp.hstack([
    final_char_vectorizer.fit_transform(full_texts),
    final_char_wb_vectorizer.fit_transform(full_texts),
    final_word_vectorizer.fit_transform(full_texts),
], format="csr", dtype=np.float32)

FINAL_INPUT_DIM = X_full.shape[1]
print("X_full:", X_full.shape)

full_tfidf_loader = DataLoader(SparseIndexDataset(full_y), batch_size=BATCH_SIZE_TFIDF, shuffle=True, collate_fn=sparse_collate_fn, num_workers=0)

final_tfidf = SparseTfidfMLP(FINAL_INPUT_DIM, NUM_CLASSES).to(DEVICE)
final_tfidf_optimizer = torch.optim.AdamW(final_tfidf.parameters(), lr=LR_TFIDF, weight_decay=WEIGHT_DECAY)

for epoch in range(1, FINAL_EPOCHS_TFIDF + 1):
    print(f"Final TFIDF epoch {epoch}/{FINAL_EPOCHS_TFIDF}")
    final_tfidf.train()
    for batch in tqdm(full_tfidf_loader, desc="Final TFIDF", leave=False):
        idx = batch["indices"].numpy()
        labels = batch["labels"].to(DEVICE)
        Xb = csr_batch_to_torch_sparse(X_full[idx], DEVICE)

        final_tfidf_optimizer.zero_grad(set_to_none=True)
        logits = final_tfidf(Xb)
        loss = tfidf_criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_tfidf.parameters(), GRAD_CLIP)
        final_tfidf_optimizer.step()

torch.save({
    "epoch": FINAL_EPOCHS_TFIDF,
    "model_state_dict": final_tfidf.state_dict(),
    "vectorizers": {"char": final_char_vectorizer, "char_wb": final_char_wb_vectorizer, "word": final_word_vectorizer},
    "input_dim": FINAL_INPUT_DIM,
    "decade_to_idx": decade_to_idx,
    "idx_to_decade": idx_to_decade,
    "decades": decades,
}, FINAL_TFIDF_PATH)

print("Guardado:", FINAL_TFIDF_PATH)


In [ ]:
# =========================
# Submission final full-data ensemble
# =========================

FINAL_SUBMISSION_PATH = "submission_roberta_large_tfidf_ensemble_full_data.csv"

@torch.no_grad()
def predict_final_roberta_probs_texts(texts, batch_size=BATCH_SIZE_ROBERTA):
    final_roberta.eval()
    probs_out = []
    for text in tqdm(texts, desc="Final RoBERTa probs"):
        crops = get_eval_crops(text)
        crop_probs = []
        for start in range(0, len(crops), batch_size):
            batch_texts = crops[start:start+batch_size]
            enc = tokenizer(batch_texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt")
            input_ids = enc["input_ids"].to(DEVICE)
            attention_mask = enc["attention_mask"].to(DEVICE)
            with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type=="cuda")):
                logits = final_roberta(input_ids, attention_mask)
                probs = torch.softmax(logits, dim=1)
            crop_probs.append(probs.detach().cpu())
        probs_out.append(torch.cat(crop_probs, dim=0).mean(dim=0).numpy())
    return np.vstack(probs_out)

@torch.no_grad()
def predict_final_tfidf_probs(Xmat, batch_size=BATCH_SIZE_TFIDF):
    final_tfidf.eval()
    out = []
    for start in tqdm(range(0, Xmat.shape[0], batch_size), desc="Final TFIDF probs"):
        Xb = csr_batch_to_torch_sparse(Xmat[start:start+batch_size], DEVICE)
        logits = final_tfidf(Xb)
        out.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
    return np.vstack(out)

eval_texts = eval_df[TEXT_COL].tolist()

X_eval_final = sp.hstack([
    final_char_vectorizer.transform(eval_texts),
    final_char_wb_vectorizer.transform(eval_texts),
    final_word_vectorizer.transform(eval_texts),
], format="csr", dtype=np.float32)

final_roberta_probs = predict_final_roberta_probs_texts(eval_texts)
final_tfidf_probs = predict_final_tfidf_probs(X_eval_final)

final_probs = best_w * final_roberta_probs + (1 - best_w) * final_tfidf_probs
final_pred = final_probs.argmax(axis=1)

answers = [idx_to_decade[int(i)] for i in final_pred]

submission = pd.DataFrame({"id": eval_df["id"].values, "answer": answers})
submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

print("Guardado:", FINAL_SUBMISSION_PATH)
display(submission.head())
display(submission["answer"].value_counts().sort_index())
